# ГП4 — нормальное обучение YOLO-pose

GPU T4, Internet **On**, Persistence **On**, секрет `WANDB_API_KEY`, zip картинок ГП2.

Не 20 эпох @ 640. Здесь **100 эпох, imgsz=1280** (на T4 обычно 2–5 часов). Ноутбук не закрывать.

Код: `git clone` ветки `missing-ml-2026`. Eval только **val** (последние 20% кадров).


In [ ]:
%pip install -q ultralytics pyarrow opencv-python-headless wandb hydra-core omegaconf pyyaml


Settings → Internet **On**. Репозиторий публичный, токен не нужен. Картинки — Add data (zip ГП2), код — только git.

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/abbos-trnv/pose_estimation.git"
BRANCH = "missing-ml-2026"
REPO = Path("/kaggle/working/pose_estimation")

def _git(*args):
    subprocess.check_call(["git", *args])

if (REPO / ".git").exists():
    _git("-C", str(REPO), "fetch", "origin", BRANCH)
    _git("-C", str(REPO), "checkout", BRANCH)
    _git("-C", str(REPO), "pull", "--ff-only", "origin", BRANCH)
else:
    _git("clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(REPO))

sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)
print("REPO", REPO, "HEAD")
subprocess.check_call(["git", "log", "-1", "--oneline"])

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
except Exception as e:
    print("wandb secret", e)


In [ ]:
!python scripts/export_yolo_pose.py data.max_frames=null


In [ ]:
!python scripts/train_pose.py


In [ ]:
from pathlib import Path
w = Path("runs/train/gp4_s1280/weights/best.pt")
print("eval weights", w, "exists", w.exists())
if not w.exists():
    raise FileNotFoundError(w)


In [ ]:
import subprocess
cmd = [
    "python",
    "scripts/eval_pose.py",
    f"model.weights={w}",
    "data=waymo_full",
    "data.subset=val",
    "protocol=both",
]
print(cmd)
subprocess.check_call(cmd)
